In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

In [2]:
# Load your dataset (replace with your actual data)
data = pd.read_csv("ade_classification.csv")

# Preprocess the text data
def preprocess_text(text):
    # Add more preprocessing steps as needed (e.g., stemming, lemmatization)
    text = text.lower()
    # ... other preprocessing steps
    return text

data['text'] = data['text'].apply(preprocess_text)

In [3]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data['text'], data['label'],test_size=0.2, random_state=42)

In [4]:

# Feature extraction using TF-IDF
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [5]:
# Print shapes for verification
print(f"X_train_tfidf shape: {X_train_tfidf.shape}")
print(f"y_train shape: {y_train.shape}")

# Check if shapes match (should be equal)
if X_train_tfidf.shape[0] != y_train.shape[0]:
    raise ValueError("X_train_tfidf and y_train have different number of samples!")

X_train_tfidf shape: (18812, 15962)
y_train shape: (18812,)


In [6]:
# Apply SMOTE on the transformed data
sm = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = sm.fit_resample(X_train_tfidf, y_train)


In [7]:
# Train a Naive Bayes classifier
clf = MultinomialNB()
clf.fit(X_train_resampled, y_train_resampled)

MultinomialNB()

In [8]:
# Make predictions
y_pred = clf.predict(X_test_tfidf)

# Evaluate the model
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.80      0.86      3325
           1       0.64      0.87      0.74      1379

    accuracy                           0.82      4704
   macro avg       0.79      0.84      0.80      4704
weighted avg       0.85      0.82      0.83      4704

[[2657  668]
 [ 178 1201]]


In [10]:
import pickle

# Save the model
with open('adverse_event_model.pkl', 'wb') as f:
    pickle.dump(clf, f)

# Save the vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

# testing Code

In [11]:
def predict_adverse_event(text_input, model, vectorizer):
  """Predicts if the given text input indicates an adverse event.

  Args:
    text_input: The text input to be classified.
    model: The trained model.
    vectorizer: The TF-IDF vectorizer used for feature extraction.

  Returns:
    The predicted class (0 or 1) and the probability of the predicted class.
  """

  # Preprocess the input text
  preprocessed_text = preprocess_text(text_input)

  # Transform the text into TF-IDF features
  X_test = vectorizer.transform([preprocessed_text])

  # Make prediction
  y_pred = model.predict(X_test)
  y_prob = model.predict_proba(X_test)

  return y_pred[0], y_prob[0][y_pred[0]]

# Example usage:
text_input = "I'm tolerating the medication well"
predicted_class, probability = predict_adverse_event(text_input, clf, vectorizer)

if predicted_class == 1:
  print("The input text indicates a potential adverse event with a probability of {:.2f}".format(probability))
else:
  print("The input text does not indicate an adverse event.")

The input text does not indicate an adverse event.
